In [ ]:
# ===== Settings =====
DATASETS = ['nadig25hepg2']
FOLDS = [0, 1, 2, 3, 4]
MODELS = ['presage']
RUN_ID = 'retrain_v1'
OPENAI_API_KEY = ''   # not needed for PRESAGE

In [ ]:
import os, sys, json, time, subprocess, shutil, hashlib, pathlib
from google.colab import drive
drive.mount('/content/drive')
OUT = pathlib.Path('/content/drive/MyDrive/wmse_colab') / RUN_ID
OUT.mkdir(parents=True, exist_ok=True)

def log(msg):
    line = time.strftime('%Y-%m-%d %H:%M:%S') + '  ' + str(msg)
    print(line, flush=True)
    with open(OUT / 'progress.log', 'a') as fh:
        fh.write(line + '\n')

def sh(cmd, cwd=None, env=None, logfile=None, check=True):
    t0 = time.time()
    shell = isinstance(cmd, str)
    if logfile:
        with open(logfile, 'a') as fh:
            fh.write('\n$ ' + (cmd if shell else ' '.join(map(str, cmd))) + '\n'); fh.flush()
            r = subprocess.run(cmd, cwd=cwd, env=env, stdout=fh, stderr=subprocess.STDOUT, shell=shell)
    else:
        r = subprocess.run(cmd, cwd=cwd, env=env, shell=shell)
    if check and r.returncode != 0:
        raise RuntimeError(f'command failed (exit {r.returncode}): {cmd}')
    return r.returncode, time.time() - t0

gpu = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], capture_output=True, text=True).stdout.strip()
log(f'== start {RUN_ID}; GPU: {gpu}; datasets={DATASETS} folds={FOLDS} models={MODELS}')
assert gpu, 'no GPU: choose a GPU runtime'

In [ ]:
# ===== Authors' code and data (public S3 bucket, checked by SHA-256) =====
REPO = pathlib.Path('/content/cellsimbench')
COMMIT = '15361aec8392fb45a75c8c9da88ca5e6510fca97'
if not REPO.exists():
    sh(['git', 'clone', '-q', 'https://github.com/shiftbioscience/Perturbation-Models-Outperform-Baselines.git', str(REPO)])
sh(['git', '-C', str(REPO), 'checkout', '-q', COMMIT])
log('author code at ' + subprocess.run(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], capture_output=True, text=True).stdout.strip())

S3 = 'https://public-access-bucket-i9sdj34j.s3.amazonaws.com/cellsimbench-data/'
SHA = {'replogle22k562': '594ddf179a743549ebc01a40cc4b6d25d8bde9d2472e89d2d613e5261690d80e',
       'nadig25hepg2': '4faf489ce12a4b9b90fc7d1415be89e167c058c0812d260b17971caeaec68525'}

def sha256(p):
    h = hashlib.sha256()
    with open(p, 'rb') as fh:
        for b in iter(lambda: fh.read(1 << 24), b''):
            h.update(b)
    return h.hexdigest()

sh('apt-get -qq update > /dev/null 2>&1; apt-get -qq install -y aria2 > /dev/null 2>&1', check=False)
for ds in DATASETS:
    p = pathlib.Path(f'/content/data/{ds}/{ds}_processed_complete.h5ad')
    p.parent.mkdir(parents=True, exist_ok=True)
    if not p.exists():
        _, dt = sh(['aria2c', '-q', '-x', '16', '-s', '16', '--file-allocation=none', '-d', str(p.parent), '-o', p.name + '.part',
                    S3 + f'data/{ds}/{ds}_processed_complete.h5ad'])
        os.replace(str(p) + '.part', p)
        log(f'{ds}: downloaded {p.stat().st_size / 1e9:.2f} GB in {dt / 60:.1f} min')
    h = sha256(p)
    assert h == SHA[ds], f'{ds} SHA-256 mismatch: {h}'
    log(f'{ds}: SHA-256 OK')

In [ ]:
# ===== Two Python environments, following each model's Dockerfile =====
ELOG = OUT / 'env_setup.log'
sh([sys.executable, '-m', 'pip', 'install', '-q', 'uv'], logfile=ELOG)
UV = shutil.which('uv') or '/usr/local/bin/uv'
ENV_MAIN, ENV_PRESAGE = '/content/env_main', '/content/env_presage'
PY_MAIN, PY_PRESAGE = ENV_MAIN + '/bin/python', ENV_PRESAGE + '/bin/python'

if not os.path.exists(ENV_MAIN + '/.ready'):
    # fMLP / scLambda Dockerfiles: pytorch 2.5.1 + CUDA 12.1, then CellSimBench, then model requirements
    sh([UV, 'venv', '-q', '--seed', '--python', '3.11', ENV_MAIN], logfile=ELOG)
    sh([UV, 'pip', 'install', '-q', '--python', PY_MAIN, 'torch==2.5.1', '--index-url', 'https://download.pytorch.org/whl/cu121'], logfile=ELOG)
    sh([UV, 'pip', 'install', '-q', '--python', PY_MAIN, '-e', str(REPO)], logfile=ELOG)
    sh([UV, 'pip', 'install', '-q', '--python', PY_MAIN, '-r', str(REPO / 'docker/fmlp/requirements.txt'),
        '-r', str(REPO / 'docker/sclambda/requirements_sclambda.txt')], logfile=ELOG)
    sh([PY_MAIN, '-c', 'import torch, cellsimbench, scanpy; print(torch.__version__, torch.cuda.is_available())'], logfile=ELOG)
    pathlib.Path(ENV_MAIN + '/.ready').touch()
log('env_main ready')

if 'presage' in MODELS and not os.path.exists(ENV_PRESAGE + '/.ready'):
    # PRESAGE Dockerfile: conda env (python 3.11, torch 2.2.0 cu121, pyg 2.5.2, lightning 2.5.1),
    # then CellSimBench with its dependencies, then force numpy<2 and anndata>=0.12
    sh([UV, 'venv', '-q', '--seed', '--python', '3.11', ENV_PRESAGE], logfile=ELOG)
    sh([UV, 'pip', 'install', '-q', '--python', PY_PRESAGE, 'torch==2.2.0', '--index-url', 'https://download.pytorch.org/whl/cu121'], logfile=ELOG)
    sh([UV, 'pip', 'install', '-q', '--python', PY_PRESAGE, 'torch-cluster==1.6.3+pt22cu121', 'pyg-lib==0.4.0+pt22cu121',
        '-f', 'https://data.pyg.org/whl/torch-2.2.0+cu121.html'], logfile=ELOG)
    sh([UV, 'pip', 'install', '-q', '--python', PY_PRESAGE, 'torch-geometric==2.5.2', 'pytorch-lightning==2.5.1.post0',
        'torchmetrics==1.7.2', 'scanpy==1.10.4', 'scikit-learn==1.6.1', 'pandas==2.2.3', 'scipy==1.15.2', 'statsmodels',
        'h5py', 'pyyaml', 'matplotlib', 'seaborn'], logfile=ELOG)
    sh([PY_PRESAGE, '-m', 'pip', 'install', '-q', '-e', str(REPO)], logfile=ELOG)
    sh([PY_PRESAGE, '-m', 'pip', 'install', '-q', 'numpy<2', 'anndata>=0.12.0'], logfile=ELOG)
    pathlib.Path(ENV_PRESAGE + '/.ready').touch()
if 'presage' in MODELS:
    log('env_presage ready')

In [ ]:
# ===== Model-specific assets =====
PLOG = OUT / 'assets.log'
if 'sclambda' in MODELS:
    if OPENAI_API_KEY:
        os.environ['OPENAI_API_KEY'] = OPENAI_API_KEY
        log('scLambda: OpenAI embeddings (original method)')
    else:
        os.environ['OPENAI_API_KEY'] = 'sk-placeholder-not-used'   # embeddings come from the cached pickle below
        builder = '''
import sys, pickle, pathlib, h5py, numpy as np
try:
    from anndata.io import read_elem
except ImportError:
    from anndata.experimental import read_elem
ds = sys.argv[1]
with h5py.File(f'/content/data/{ds}/{ds}_processed_complete.h5ad', 'r') as f:
    E = read_elem(f['uns/embeddings_genept'])
    cond = read_elem(f['obs/condition'])
genes = sorted({g for c in set(map(str, cond)) if c not in ('control', 'ctrl') for g in c.replace('+ctrl', '').split('+')})
emb = {g: np.asarray(E.loc[g].values, dtype=np.float32) for g in genes if g in E.index}
out = pathlib.Path(f'/content/sclambda_cache/{ds}'); out.mkdir(parents=True, exist_ok=True)
pickle.dump(emb, open(out / 'gene_embeddings.pkl', 'wb'))
print(f'{ds}: {len(genes)} perturbed genes, {len(emb)} with GenePT embedding, dim {E.shape[1]}')
'''
        pathlib.Path('/content/build_genept_pkl.py').write_text(builder)
        for ds in DATASETS:
            if not pathlib.Path(f'/content/sclambda_cache/{ds}/gene_embeddings.pkl').exists():
                sh([PY_MAIN, '/content/build_genept_pkl.py', ds], logfile=PLOG)
        log('scLambda: GenePT embeddings cached (documented deviation) -- ' + open(PLOG).read().strip().splitlines()[-1])

if 'presage' in MODELS:
    if not os.path.exists('/presage_src'):
        sh(['git', 'clone', '-q', 'https://github.com/Genentech/PRESAGE.git', '/content/presage_repo'], logfile=PLOG)
        sh(['git', '-C', '/content/presage_repo', 'checkout', '-q', '2c7b231c60cf110c4aab0acf8ecd2c2b3268fffb'], logfile=PLOG)
        shutil.copytree('/content/presage_repo/src', '/presage_src')
        shutil.copytree('/content/presage_repo/sample_files', '/presage_src/sample_files')
    PW = pathlib.Path('/content/presage_work'); PW.mkdir(exist_ok=True)
    if not (PW / 'cache').exists():
        sh(['aria2c', '-q', '-x', '8', '-s', '8', '--file-allocation=none', '-d', str(PW), '-o', 'cache.tar.gz',
            'https://zenodo.org/records/15587986/files/cache.tar.gz?download=1'], logfile=PLOG)
        sh(['tar', '--no-same-owner', '-xzf', str(PW / 'cache.tar.gz'), '-C', str(PW)], logfile=PLOG)
        os.remove(PW / 'cache.tar.gz')
    for fn in ['presage_wrapper.py', 'run_model.py', 'utils.py', 'cellsimbench_datamodule.py']:
        shutil.copy(REPO / 'docker/presage' / fn, PW / fn)
    sh([PY_PRESAGE, '-c', "import sys; sys.path.insert(0, '/presage_src'); import presage, model_harness, presage_datamodule; "
        "from torch_geometric.nn.models import Node2Vec; import cellsimbench.core.data_manager; print('presage imports ok')"], logfile=PLOG)
    log('PRESAGE assets ready; cache: ' + ', '.join(sorted(os.listdir(PW / 'cache'))[:8]))

In [ ]:
%%writefile /content/wmse_runner.py
"""Train / predict / export one (model, dataset, fold) the way CellSimBench's docker runners do."""
import argparse, hashlib, json, pathlib, subprocess, sys
from omegaconf import OmegaConf

ap = argparse.ArgumentParser()
ap.add_argument('--model', required=True); ap.add_argument('--dataset', required=True)
ap.add_argument('--fold', type=int, required=True); ap.add_argument('--stage', choices=['train', 'predict', 'export'], required=True)
a = ap.parse_args()
REPO = pathlib.Path('/content/cellsimbench')
data_path = f'/content/data/{a.dataset}/{a.dataset}_processed_complete.h5ad'
run = pathlib.Path(f'/content/runs/{a.dataset}/{a.model}/fold{a.fold}'); run.mkdir(parents=True, exist_ok=True)
split = f'split_fold_{a.fold}'
hp = OmegaConf.to_object(OmegaConf.load(REPO / f'cellsimbench/configs/model/{a.model}.yaml'))['hyperparameters']
if a.model == 'sclambda':
    hp['gene_embedding_config']['cache_dir'] = f'/content/sclambda_cache/{a.dataset}'

if a.stage in ('train', 'predict'):
    cfile = run / 'conditions.json'
    if not cfile.exists():   # same lists TrainingRunner/ModelRunner pass to the containers
        from cellsimbench.core.data_manager import DataManager
        dcfg = OmegaConf.to_object(OmegaConf.load(REPO / f'cellsimbench/configs/dataset/{a.dataset}.yaml'))
        dcfg['data_path'] = data_path
        dm = DataManager(dcfg)
        import h5py
        try:
            from anndata.io import read_elem
        except ImportError:
            from anndata.experimental import read_elem
        with h5py.File(data_path, 'r') as fh:
            obs = read_elem(fh['obs'])
        json.dump(dm.get_perturbation_conditions(split, obs), open(cfile, 'w'))
    conds = json.load(open(cfile))
    if a.stage == 'train':
        cfg = {'mode': 'train', 'data_path': data_path, 'split_name': split, 'covariate_key': 'donor_id',
               'hyperparameters': hp, 'output_dir': str(run) + '/', 'checkpoint_dir': str(run / 'checkpoints') + '/',
               'train_conditions': conds['train'], 'val_conditions': conds['val'], 'test_conditions': conds['test']}
    else:
        cfg = {'mode': 'predict', 'data_path': data_path, 'model_path': str(run) + '/', 'split_name': split,
               'test_conditions': conds['test'], 'covariate_key': 'donor_id', 'hyperparameters': hp,
               'output_path': str(run / 'predictions.h5ad')}
    cpath = run / f'{a.stage}_config.json'
    json.dump(cfg, open(cpath, 'w'), indent=1)
    sys.exit(subprocess.run([sys.executable, 'run_model.py', a.stage, str(cpath)]).returncode)

# export: float32 parts below 4 MB each, plus metadata, for transfer to the project folder
import anndata as ad, numpy as np
A = ad.read_h5ad(run / 'predictions.h5ad')
X = A.X.toarray() if hasattr(A.X, 'toarray') else np.asarray(A.X)
X = X.astype(np.float32)
meta = {'shape': list(X.shape), 'obs_names': [str(x) for x in A.obs_names], 'var_names': [str(x) for x in A.var_names],
        'obs': {c: [str(v) for v in A.obs[c]] for c in A.obs.columns}, 'parts': [],
        'predictions_h5ad_sha256': hashlib.sha256((run / 'predictions.h5ad').read_bytes()).hexdigest()}
rows = max(1, 4_000_000 // (X.shape[1] * 4))
for i in range(0, X.shape[0], rows):
    fn = run / f'pred_part{i // rows:02d}.npy'
    np.save(fn, X[i:i + rows])
    meta['parts'].append({'file': fn.name, 'rows': int(min(rows, X.shape[0] - i)), 'sha256': hashlib.sha256(fn.read_bytes()).hexdigest()})
json.dump(meta, open(run / 'pred_meta.json', 'w'))
print('exported', X.shape, len(meta['parts']), 'parts')

In [ ]:
# ===== Train, predict and export each (dataset, fold, model); finished runs are skipped, failures do not stop the rest =====
PY = {'fmlp_genept': PY_MAIN, 'sclambda': PY_MAIN, 'presage': PY_PRESAGE}
CWD = {'fmlp_genept': str(REPO / 'docker/fmlp'), 'sclambda': str(REPO / 'docker/sclambda'), 'presage': '/content/presage_work'}
results = {}
for ds in DATASETS:
    for k in FOLDS:
        for m in MODELS:
            tag = f'{ds}/{m}/fold{k}'
            run = pathlib.Path(f'/content/runs/{tag}'); run.mkdir(parents=True, exist_ok=True)
            dst = OUT / tag; dst.mkdir(parents=True, exist_ok=True)
            if (dst / 'pred_meta.json').exists():
                log(f'{tag}: already done, skipped'); results[tag] = 'ok (earlier)'; continue
            env = dict(os.environ, PYTHONUNBUFFERED='1', PYTHONPATH='/presage_src' if m == 'presage' else '')
            if m == 'presage':   # fresh-container semantics: rebuild the gene-knowledge tensor for this dataset/fold
                stale = list(pathlib.Path('/content/presage_work/cache/pathway_embeddings').glob('WeightedDeepset.*.embeddings.pkl'))
                for f in stale:
                    f.unlink()
                log(f'{tag}: cleared {len(stale)} cached WeightedDeepset tensor(s) before training')
            try:
                for stage in ('train', 'predict', 'export'):
                    _, dt = sh([PY[m], '/content/wmse_runner.py', '--model', m, '--dataset', ds, '--fold', str(k), '--stage', stage],
                               cwd=CWD[m], env=env, logfile=run / 'run.log')
                    log(f'{tag}: {stage} done in {dt / 60:.1f} min')
                for f in ['predictions.h5ad', 'pred_meta.json', 'conditions.json', 'train_config.json', 'predict_config.json']:
                    shutil.copy(run / f, dst / f)
                for f in sorted(run.glob('pred_part*.npy')):
                    shutil.copy(f, dst / f.name)
                results[tag] = 'ok'
            except Exception as e:
                results[tag] = f'FAILED: {e}'
                log(f'{tag}: FAILED -- {e}')
            finally:
                if (run / 'run.log').exists():
                    shutil.copy(run / 'run.log', dst / 'run.log')
log('== finished: ' + json.dumps(results, ensure_ascii=False))